# Retrieval Chain Without LCEL

This notebook implements a retrieval QA pipeline step by step using `RetrievalQA.from_chain_type` (no LCEL composition).

## Block 0: Project Setup and Environment Loading

In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'ingestion.py').exists():
    PROJECT_DIR = Path('/Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer')

load_dotenv(PROJECT_DIR / '.env')
print('Loaded .env from:', PROJECT_DIR / '.env')
print('INDEX_NAME:', os.getenv('INDEX_NAME'))

Loaded .env from: /Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer/.env
INDEX_NAME: medium-analyer


## Block 1: Validate Required API and Index Configuration

In [2]:
required_keys = ['OPENAI_API_KEY', 'PINECONE_API_KEY', 'INDEX_NAME']
missing = [k for k in required_keys if not os.getenv(k)]
if missing:
    raise ValueError(f'Missing required environment keys: {missing}')
print('All required keys are present.')

All required keys are present.


## Block 2: Create Embeddings and Connect Vector Store

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(openai_api_key=os.environ['OPENAI_API_KEY'])
vectorstore = PineconeVectorStore(index_name=os.environ['INDEX_NAME'], embedding=embeddings)
print('Connected to Pinecone index:', os.environ['INDEX_NAME'])

/Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to Pinecone index: medium-analyer


## Block 3: Build Retriever and Preview Retrieved Context

In [4]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
preview_question = 'What is this article mainly about?'
preview_docs = retriever.invoke(preview_question)
print('Retrieved docs:', len(preview_docs))
if preview_docs:
    print('First doc preview:', preview_docs[0].page_content[:220].replace('\n', ' '), '...')

Retrieved docs: 4
First doc preview: Milvus docs Here are some of the features of Milvus: ...


## Block 4: Define QA Prompt Template

In [5]:
from langchain_core.prompts import PromptTemplate

qa_prompt = PromptTemplate(
    input_variables=['context', 'question'],
    template=(
        'You are a precise assistant for article analysis.\n'
        'Answer only from the provided context.\n'
        'If context is insufficient, say so clearly.\n\n'
        'Context:\n{context}\n\n'
        'Question: {question}\n\n'
        'Answer:'
    )
)
print('Prompt template ready.')

Prompt template ready.


## Block 5: Construct `retrieval_chain_without_lcel`

In [6]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
retrieval_chain_without_lcel = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={'prompt': qa_prompt},
)

print('retrieval_chain_without_lcel initialized.')

retrieval_chain_without_lcel initialized.


## Block 6: Run Query and Print Answer with Sources

In [7]:
question = 'Summarize the key points of the article in 5 bullet points.'
result = retrieval_chain_without_lcel.invoke({'query': question})

print('Question:', question)
print('\nAnswer:\n')
print(result.get('result', ''))

source_docs = result.get('source_documents', [])
print(f'\nSource documents used: {len(source_docs)}')
for i, doc in enumerate(source_docs[:3], start=1):
    snippet = doc.page_content[:180].replace('\n', ' ')
    print(f'- Source {i}: {snippet}...')

Question: Summarize the key points of the article in 5 bullet points.

Answer:

The provided context does not contain specific details about the content of the articles mentioned. Therefore, I cannot summarize the key points.

Source documents used: 4
- Source 1: Milvus docs Here are some of the features of Milvus:...
- Source 2: Milvus docs Here are some of the features of Milvus:...
- Source 3: Milvus docs Here are some of the features of Milvus:...


## Block 7: Optional Second Query

In [8]:
question_2 = 'What technologies or tools are discussed?'
result_2 = retrieval_chain_without_lcel.invoke({'query': question_2})
print('Question:', question_2)
print('\nAnswer:\n')
print(result_2.get('result', ''))

Question: What technologies or tools are discussed?

Answer:

The context discusses vector databases, LLM (Large Language Model) applications, and their integration capabilities, query interfaces, and deployment options. However, specific technologies or tools are not mentioned.
